In [1]:
"""
generate_html.py  —  reads sw_solution.json, writes visual.html
"""

import json, math

# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Which statistical trend filter is shown by default in the output gap chart.
# Options: "hp_full"  — Full-information HP (always runs on HP_FULL_WINDOW quarters,
#                       two-sided smoother, best for illustrating the "textbook" trend)
#          "hp_rt"    — Real-time HP (recursive: at period t only uses data 0..t,
#                       illustrates real-time estimation uncertainty)
#          "kalman"   — Kalman filter (local-level model, one-sided real-time,
#                       optimal under its own Gaussian assumptions)
DEFAULT_STAT_FILTER = "hp_full"

# Number of pre-shock SS observations prepended before any filter runs.
# All three filters see HP_PAD zeros before the shock arrives, giving them
# a well-defined anchor and making their outputs comparable at t=0.
HP_PAD = 10

# Fixed window for full-information HP (quarters).
# The filter always runs on this many quarters regardless of the horizon slider,
# so the trend estimate is stable as the user drags the horizon.
HP_FULL_WINDOW = 80
# ──────────────────────────────────────────────────────────────────────────────

with open("sw_solution.json") as f:
    sol = json.load(f)

# ── Build JS-ready data blob ───────────────────────────────────────────────
ghx         = sol["ghx"]
ghu         = sol["ghu"]
state_rows  = sol["state_rows"]
VI          = sol["VI"]
shock_keys  = sol["shock_keys"]
shock_labels= sol["shock_labels"]
shock_idx   = sol["shock_js_idx"]
model_rhos  = sol["model_rhos"]
p           = sol["params"]

shocks_js = [
    {"key": shock_keys[i], "label": shock_labels[i],
     "idx": shock_idx[i],  "rho": model_rhos[i]}
    for i in range(len(shock_keys))
]

# Steady-state values for rates
beta_real     = 0.9926
r_star_real   = (1/beta_real - 1) * 100
pi_target_ann = p["constepinf"] * 4
r_nom_ss      = r_star_real + pi_target_ann

data = {
    "ghx": ghx, "ghu": ghu, "state_rows": state_rows, "VI": VI,
    "shocks": shocks_js, "params": p,
    "r_star_real": r_star_real, "r_nom_ss": r_nom_ss,
    # Filter config passed through to JS
    "default_filter":  DEFAULT_STAT_FILTER,
    "hp_pad":          HP_PAD,
    "hp_full_window":  HP_FULL_WINDOW,
}
data_json  = json.dumps(data, separators=(",", ":"))
ctrend     = p["ctrend"]
constepinf = p["constepinf"]

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>SW2007 — IRF Explorer</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
:root{{
  --bg:#fff;--bg2:#f5f4f0;--bg3:#eeecea;
  --border:rgba(0,0,0,.09);--border2:rgba(0,0,0,.17);
  --text:#1a1a18;--text2:#5f5e5a;--text3:#888780;
  /* Semantic colour palette (consistent across all charts):
       DK = dark grey  → steady-state / BGP reference lines
       BL = blue       → NK sticky-price economy
       OR = orange     → flex-price economy & gap from flex prices
       RD = red        → statistical trend (HP / Kalman) & gap from it  */
  --dk:#3d3d3a;--bl:#2F7CC9;--or:#D4720A;--rd:#C0392B;
  --r:8px;--rl:12px;
}}
@media(prefers-color-scheme:dark){{
  :root{{--bg:#1c1c1a;--bg2:#252523;--bg3:#2e2e2b;
        --border:rgba(255,255,255,.09);--border2:rgba(255,255,255,.17);
        --text:#e8e6df;--text2:#b4b2a9;--text3:#888780;
        --dk:#aaa9a4;--bl:#5FA8E8;--or:#E8962A;--rd:#E05C4E}}
}}
body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      background:var(--bg3);color:var(--text);font-size:14px;line-height:1.5}}
.page{{max-width:1140px;margin:0 auto;padding:2rem 1.25rem 4rem}}
h1{{font-size:19px;font-weight:500;margin-bottom:3px}}
.sub{{font-size:12px;color:var(--text2);margin-bottom:1.4rem}}
.card{{background:var(--bg);border:.5px solid var(--border);
       border-radius:var(--rl);padding:1.1rem 1.25rem;margin-bottom:.9rem}}
.card-title{{font-size:11px;font-weight:500;color:var(--text3);
             text-transform:uppercase;letter-spacing:.06em;margin-bottom:.85rem}}
.shock-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));gap:10px}}
.shock-block{{background:var(--bg2);border-radius:var(--r);padding:10px 12px}}
.shock-head{{display:flex;align-items:center;gap:10px;margin-bottom:8px}}
.tog{{position:relative;width:34px;height:19px;flex-shrink:0;cursor:pointer}}
.tog input{{opacity:0;width:0;height:0}}
.tog-track{{position:absolute;inset:0;background:var(--border2);border-radius:10px;transition:.18s}}
.tog input:checked+.tog-track{{background:var(--bl)}}
.tog-thumb{{position:absolute;width:13px;height:13px;background:#fff;
            border-radius:50%;top:3px;left:3px;transition:.18s}}
.tog input:checked~.tog-thumb{{left:18px}}
.shock-name{{flex:1;font-size:13px}}
.shock-val{{font-size:12px;color:var(--text2);font-variant-numeric:tabular-nums;min-width:36px;text-align:right}}
.sl-row{{display:flex;justify-content:space-between;font-size:11px;
         color:var(--text3);margin-bottom:3px}}
input[type=range]{{width:100%;-webkit-appearance:none;height:3px;border-radius:2px;
                   background:var(--bg3);outline:none;cursor:pointer;margin-bottom:7px}}
input[type=range]::-webkit-slider-thumb{{-webkit-appearance:none;width:14px;height:14px;
  border-radius:50%;background:var(--bl);border:2px solid var(--bg)}}
input[type=range]::-moz-range-thumb{{width:12px;height:12px;border-radius:50%;
  background:var(--bl);border:2px solid var(--bg)}}
.settings{{display:flex;gap:10px;flex-wrap:wrap;align-items:flex-end;margin-bottom:.9rem}}
.ctrl{{background:var(--bg2);border-radius:var(--r);padding:9px 12px;min-width:160px}}
.ctrl-head{{display:flex;justify-content:space-between;margin-bottom:6px}}
.ctrl-name{{font-size:12px;color:var(--text2)}}
.ctrl-val{{font-size:14px;font-weight:500;font-variant-numeric:tabular-nums}}
.ctrl-label{{font-size:12px;color:var(--text2);margin-bottom:6px}}
.filter-tabs{{display:flex;gap:4px}}
.ftab{{font-size:11px;padding:5px 11px;border-radius:6px;cursor:pointer;
        border:1px solid var(--border2);background:transparent;
        color:var(--text2);transition:.15s;white-space:nowrap}}
.ftab.active{{background:var(--bl);color:#fff;border-color:var(--bl);font-weight:500}}
.legend{{display:flex;gap:16px;flex-wrap:wrap;font-size:12px;color:var(--text2);margin-bottom:.5rem}}
.leg{{display:flex;align-items:center;gap:5px}}
.leg-swatch{{width:20px;height:2px;border-radius:1px;flex-shrink:0}}
.leg-swatch.dash{{background:none!important;border-top:2px dashed}}
.leg-swatch.dot{{background:none!important;border-top:2px dotted}}
.legend-note{{font-size:10px;color:var(--text3);font-style:italic;margin-bottom:.85rem;line-height:1.6}}
.chart-grid{{display:grid;grid-template-columns:1fr 1fr;gap:12px}}
.chart-card{{background:var(--bg);border:.5px solid var(--border);
             border-radius:var(--rl);padding:12px 14px}}
.chart-name{{font-size:13px;font-weight:500;margin-bottom:2px}}
.chart-sub{{font-size:11px;color:var(--text3);margin-bottom:8px}}
.chart-wrap{{position:relative;height:220px}}
.eq{{font-size:11px;color:var(--text3);line-height:1.9}}
@media(max-width:650px){{
  .chart-grid{{grid-template-columns:1fr}}
  .page{{padding:1rem}}
}}
</style>
</head>
<body>
<div class="page">

<h1>Smets-Wouters (2007) — Impulse Response Explorer</h1>
<p class="sub">Select any combination of shocks. IRFs are log-deviations from balanced growth path.
Deterministic trend ({ctrend:.3f}%/qtr, {ctrend*4:.2f}%/yr) and inflation target ({constepinf:.2f}%/qtr, {constepinf*4:.1f}%/yr) shown in output chart.</p>

<div class="card">
  <div class="card-title">Shocks — select any combination</div>
  <div class="shock-grid" id="shock-grid"></div>
</div>

<div class="settings">
  <div class="ctrl">
    <div class="ctrl-head">
      <span class="ctrl-name">Horizon (quarters)</span>
      <span class="ctrl-val" id="v-T">16</span>
    </div>
    <input type="range" id="sl-T" min="8" max="80" step="2" value="16" oninput="update()">
  </div>
  <div class="ctrl">
    <div class="ctrl-label">Statistical trend filter</div>
    <div class="filter-tabs">
      <button class="ftab" id="ftab-hp_full" onclick="setFilter('hp_full')">HP full-info</button>
      <button class="ftab" id="ftab-hp_rt"   onclick="setFilter('hp_rt')">HP real-time</button>
      <button class="ftab" id="ftab-kalman"  onclick="setFilter('kalman')">Kalman</button>
    </div>
  </div>
</div>

<div class="legend">
  <div class="leg"><div class="leg-swatch" style="background:var(--dk)"></div>Steady-state / BGP</div>
  <div class="leg"><div class="leg-swatch" style="background:var(--bl)"></div>NK sticky-price economy</div>
  <div class="leg"><div class="leg-swatch dash" style="border-color:var(--or)"></div>Flex-price economy (y*, r*)</div>
  <div class="leg"><div class="leg-swatch dot"  style="border-color:var(--rd)"></div>Statistical trend (HP / Kalman)</div>
</div>
<div class="legend" style="margin-top:-.2rem">
  <div class="leg"><div class="leg-swatch" style="background:var(--or)"></div>Gap: y − y* (NK minus flex)</div>
  <div class="leg"><div class="leg-swatch" style="background:var(--rd)"></div>Gap: y − statistical trend</div>
  <div class="leg"><div class="leg-swatch" style="background:var(--dk)"></div>Gap: y − BGP (deviation from SS growth path)</div>
</div>
<p class="legend-note">
  HP real-time &amp; Kalman: at each period t only data 0…t is used (pre-seeded with {HP_PAD} SS quarters for comparability).
  Full-info HP always filters {HP_FULL_WINDOW} quarters regardless of the horizon slider.
  Shock re-application ρ: the same shock re-hits each period with geometrically decaying magnitude (on top of the model's own AR dynamics).
</p>

<div class="chart-grid">

  <div class="chart-card">
    <div class="chart-name">Output (log level, %)</div>
    <div class="chart-sub">Dark: BGP · Blue: NK output · Orange dashed: flex y* · Red dotted: statistical trend</div>
    <div class="chart-wrap"><canvas id="c-y"></canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Output gap (%)</div>
    <div class="chart-sub">Orange: y − y* · Red: y − stat. trend · Dark: y − BGP</div>
    <div class="chart-wrap"><canvas id="c-gap"></canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Inflation (deviation from target, pp ann.)</div>
    <div class="chart-sub">Blue: NK inflation deviation from target · Dark dashed: zero</div>
    <div class="chart-wrap"><canvas id="c-pi"></canvas></div>
  </div>

  <div class="chart-card">
    <div class="chart-name">Interest rates (deviation from SS, pp ann.)</div>
    <div class="chart-sub">Blue: nominal rate · Orange dashed: natural rate r*</div>
    <div class="chart-wrap"><canvas id="c-r"></canvas></div>
  </div>

</div>

<div class="card" style="margin-top:.9rem">
  <div class="card-title">Model</div>
  <div class="eq">
    Smets &amp; Wouters (2007) — medium-scale NK DSGE · Capital accumulation · Investment adjustment costs · Capital utilization<br>
    Habits λ={p['chabb']:.2f} · σ={p['csigma']:.2f} · Calvo prices ξ_p={p['cprobp']:.2f} · Calvo wages ξ_w={p['cprobw']:.2f}<br>
    Taylor: ρ={p['crr']:.3f} · φ_π={p['crpi']:.3f} · φ_y={p['cry']:.4f} · φ_Δy={p['crdy']:.4f}<br>
    Solution: Dynare first-order perturbation · Output gap = NK − flex-price output · HP λ=1600 · Kalman: local-level model (q≈1/1600, calibrated to HP-1600)
  </div>
</div>

</div>

<script>
const D = {data_json};
const GHX = D.ghx, GHU = D.ghu;
const SR  = D.state_rows;
const VI  = D.VI;
const P   = D.params;
const TREND_Q     = P.ctrend;
const PI_TGT      = P.constepinf;
const R_STAR_REAL = D.r_star_real;
const R_NOM_SS    = D.r_nom_ss;
const HP_PAD      = D.hp_pad;         // pre-shock SS quarters prepended before filtering
const HP_FULL_WIN = D.hp_full_window; // fixed window for full-info HP

let activeFilter = D.default_filter;  // "hp_full" | "hp_rt" | "kalman"

// ── HP filter core (pentadiagonal Gaussian elimination) ──────────────────────
function hpCore(y, lam=1600) {{
  const n = y.length;
  const a=new Array(n).fill(0),b=new Array(n).fill(0),
        c=new Array(n).fill(0),d=new Array(n).fill(0),
        e=new Array(n).fill(0);
  for(let i=2;i<n-2;i++){{a[i]+=lam;b[i]-=4*lam;c[i]+=6*lam;d[i]-=4*lam;e[i]+=lam;}}
  if(n>2){{c[0]+=lam;d[0]-=2*lam;e[0]+=lam;}}
  if(n>3){{b[1]-=2*lam;c[1]+=5*lam;d[1]-=4*lam;e[1]+=lam;}}
  if(n>3){{a[n-2]+=lam;b[n-2]-=4*lam;c[n-2]+=5*lam;d[n-2]-=2*lam;}}
  if(n>2){{a[n-1]+=lam;b[n-1]-=2*lam;c[n-1]+=lam;}}
  for(let i=0;i<n;i++) c[i]+=1;
  const A=Array.from({{length:n}},()=>new Array(n).fill(0));
  for(let i=0;i<n;i++){{
    if(i-2>=0) A[i][i-2]=a[i]; if(i-1>=0) A[i][i-1]=b[i];
    A[i][i]=c[i]; if(i+1<n) A[i][i+1]=d[i]; if(i+2<n) A[i][i+2]=e[i];
  }}
  const rhs=[...y];
  for(let col=0;col<n;col++){{
    let mx=col;
    for(let r=col+1;r<n;r++) if(Math.abs(A[r][col])>Math.abs(A[mx][col])) mx=r;
    [A[col],A[mx]]=[A[mx],A[col]];[rhs[col],rhs[mx]]=[rhs[mx],rhs[col]];
    const pv=A[col][col]; if(Math.abs(pv)<1e-14) continue;
    for(let r=col+1;r<n;r++){{
      const f=A[r][col]/pv;
      for(let k=col;k<n;k++) A[r][k]-=f*A[col][k];
      rhs[r]-=f*rhs[col];
    }}
  }}
  const tr=new Array(n).fill(0);
  for(let i=n-1;i>=0;i--){{
    let s=rhs[i]; for(let j=i+1;j<n;j++) s-=A[i][j]*tr[j]; tr[i]=s/A[i][i];
  }}
  return tr;
}}

// Build a level series from a log-deviation array: prepend HP_PAD SS values,
// optionally extend to fullLen, adding deterministic trend throughout.
function buildLevel(y_dev, fullLen) {{
  const T = y_dev.length;
  const total = Math.max(T, fullLen) + HP_PAD;
  const lev = new Array(total);
  for(let i=0;i<HP_PAD;i++)    lev[i] = TREND_Q*(i-HP_PAD);       // pre-shock SS
  for(let i=0;i<T;i++)         lev[HP_PAD+i] = y_dev[i]+TREND_Q*i; // shock path
  for(let i=T;i<total-HP_PAD;i++) lev[HP_PAD+i] = TREND_Q*i;      // tail zeros
  return lev;
}}

// Full-information HP: always run on HP_FULL_WIN quarters (independent of horizon).
// Returns trend in level terms for the T visualised quarters.
function hpFull(y_dev, T, lam=1600) {{
  const lev = buildLevel(y_dev, HP_FULL_WIN);
  const tr  = hpCore(lev, lam);
  return tr.slice(HP_PAD, HP_PAD+T);
}}

// Real-time HP: at each t, re-run HP on all data seen so far (pre-seeded with
// HP_PAD SS quarters). Only the last point of each run is kept — the current
// real-time trend estimate. This mimics a statistical agency updating its
// trend estimate as new data arrives.
function hpRealtime(y_dev, T, lam=1600) {{
  const result = new Array(T);
  for(let t=0;t<T;t++) {{
    const slice = new Array(HP_PAD).fill(0);
    for(let k=0;k<=t;k++) slice.push(y_dev[k]);
    const lev = slice.map((v,i)=>v+TREND_Q*(i-HP_PAD));
    const tr  = hpCore(lev, lam);
    result[t] = tr[tr.length-1];
  }}
  return result;
}}

// Kalman filter — local-level model: trend (random walk with drift) + AR(1) gap.
// Purely statistical, no connection to the DSGE model.
// Signal-to-noise ratio q≈1/1600 calibrated to approximate HP-1600 asymptotically
// (Harvey & Jaeger 1993). Pre-seeded with HP_PAD SS quarters for comparability.
// Returns trend estimates in level terms.
function kalmanFilter(y_dev, T) {{
  // Hyperparameters (normalised to obs noise = 1)
  const sig2_eta = 1/1600; // trend innovation variance
  const sig2_xi  = 0.5;    // gap innovation variance
  const sig2_eps = 1.0;    // measurement noise variance
  const rho_c    = 0.85;   // gap AR(1) coefficient

  // State [tau_dev, c] where tau_dev is trend log-deviation from BGP
  // Transition: tau_dev_t = tau_dev_{{t-1}} + eta_t
  //             c_t       = rho_c * c_{{t-1}} + xi_t
  // Measurement: y_dev_t  = tau_dev_t + c_t + eps_t

  let x  = [0, 0];                          // state mean
  let Pk = [[10, 0],[0, 10]];               // state covariance (diffuse init)

  // Warm-up: feed HP_PAD pre-shock observations (y_dev=0 on BGP)
  for(let i=0;i<HP_PAD;i++) {{
    [x,Pk] = kStep(x, Pk, 0, rho_c, sig2_eta, sig2_xi, sig2_eps);
  }}

  const trend_lev = new Array(T);
  for(let t=0;t<T;t++) {{
    [x,Pk] = kStep(x, Pk, y_dev[t], rho_c, sig2_eta, sig2_xi, sig2_eps);
    trend_lev[t] = x[0] + TREND_Q*t;  // trend in level terms
  }}
  return trend_lev;
}}

function kStep(x, Pv, y_obs, rho_c, sig2_eta, sig2_xi, sig2_eps) {{
  // Predict
  const xp = [x[0], rho_c*x[1]];
  const Pp = [
    [Pv[0][0]+sig2_eta,            rho_c*Pv[0][1]           ],
    [rho_c*Pv[1][0],   rho_c*rho_c*Pv[1][1]+sig2_xi]
  ];
  // Innovation
  const innov = y_obs - (xp[0]+xp[1]);
  const S     = Pp[0][0]+Pp[1][1]+2*Pp[0][1]+sig2_eps;
  // Kalman gain
  const K = [(Pp[0][0]+Pp[0][1])/S, (Pp[1][0]+Pp[1][1])/S];
  // Update
  const xu = [xp[0]+K[0]*innov, xp[1]+K[1]*innov];
  const Pu = [
    [Pp[0][0]-K[0]*(Pp[0][0]+Pp[0][1]), Pp[0][1]-K[0]*(Pp[0][1]+Pp[1][1])],
    [Pp[1][0]-K[1]*(Pp[0][0]+Pp[0][1]), Pp[1][1]-K[1]*(Pp[0][1]+Pp[1][1])]
  ];
  return [xu, Pu];
}}

// Dispatch to active filter; returns trend in LEVEL terms (log %).
function computeStatTrend(y_dev, T) {{
  if(activeFilter==='hp_full') return hpFull(y_dev, T);
  if(activeFilter==='hp_rt')   return hpRealtime(y_dev, T);
  if(activeFilter==='kalman')  return kalmanFilter(y_dev, T);
  return hpFull(y_dev, T);
}}

// ── IRF propagation ───────────────────────────────────────────────────────────
function propagate(shock_col, size, T) {{
  const nV=GHX.length;
  const path=Array.from({{length:T}},()=>new Array(nV).fill(0));
  for(let v=0;v<nV;v++) path[0][v]=GHU[v][shock_col]*size;
  for(let t=1;t<T;t++){{
    const sv=SR.map(r=>path[t-1][r]);
    for(let v=0;v<nV;v++){{
      let s=0; for(let k=0;k<sv.length;k++) s+=GHX[v][k]*sv[k];
      path[t][v]=s;
    }}
  }}
  return path;
}}

// Combine multiple shocks with optional re-application (rho slider).
// rho=0  → single one-off shock (default).
// rho>0  → the shock re-hits every subsequent period with magnitude
//           size * rho^s, layered on top of the model's own state dynamics.
// Note: the model's ghx already encodes any AR structure from the mod file.
function computeIRF(entries, T) {{
  const nV=GHX.length;
  const out=Array.from({{length:T}},()=>new Array(nV).fill(0));
  for(const e of entries) {{
    const base=propagate(e.shockIdx,1.0,T);
    for(let s=0;s<T;s++) {{
      const mag=e.size*Math.pow(e.rho,s);
      if(mag===0) continue;
      for(let t=0;t<T-s;t++)
        for(let v=0;v<nV;v++)
          out[s+t][v]+=base[t][v]*mag;
    }}
  }}
  return out;
}}

// ── Chart colours (respect dark mode) ────────────────────────────────────────
const dk=()=>getComputedStyle(document.documentElement).getPropertyValue('--dk').trim();
const bl=()=>getComputedStyle(document.documentElement).getPropertyValue('--bl').trim();
const or_=()=>getComputedStyle(document.documentElement).getPropertyValue('--or').trim();
const rd=()=>getComputedStyle(document.documentElement).getPropertyValue('--rd').trim();
const dark_mode=()=>window.matchMedia('(prefers-color-scheme:dark)').matches;
const gc=()=>dark_mode()?'rgba(255,255,255,.07)':'rgba(0,0,0,.06)';
const tc=()=>dark_mode()?'#777':'#999';
const zc=()=>dark_mode()?'rgba(255,255,255,.22)':'rgba(0,0,0,.18)';

function ds(data,colorFn,dash=[],width=1.8,label=''){{
  return{{data,label,borderColor:colorFn(),borderWidth:width,
          borderDash:dash,pointRadius:0,tension:0.3,fill:false}};
}}

function mkChart(id, datasets) {{
  return new Chart(document.getElementById(id),{{
    type:'line', data:{{labels:[],datasets}},
    options:{{
      responsive:true, maintainAspectRatio:false, animation:{{duration:160}},
      interaction:{{mode:'index',intersect:false}},
      plugins:{{
        legend:{{display:true,position:'top',
                 labels:{{font:{{size:10}},color:tc(),boxWidth:14,padding:8}}}},
        tooltip:{{callbacks:{{label:c=>`${{c.dataset.label||''}}: ${{c.parsed.y.toFixed(3)}}`}}}}
      }},
      scales:{{
        x:{{grid:{{color:gc()}},ticks:{{color:tc(),font:{{size:10}},maxTicksLimit:9}},
            title:{{display:true,text:'quarters',color:tc(),font:{{size:10}}}}}},
        y:{{grid:{{color:ctx=>ctx.tick.value===0?zc():gc(),
                  lineWidth:ctx=>ctx.tick.value===0?1.8:1}},
            ticks:{{color:tc(),font:{{size:10}},callback:v=>v.toFixed(2)}}}}
      }}
    }}
  }});
}}

const charts = {{
  // Output levels: BGP (dark solid), NK (blue solid), flex y* (orange dash), stat trend (red dot)
  y: mkChart('c-y',[
    ds([],dk,[8,4],1.4,'BGP trend'),
    ds([],bl,[],2.0,'NK output (sticky)'),
    ds([],or_,[6,4],1.6,'Flex-price output y*'),
    ds([],rd,[3,3],1.4,'Statistical trend'),
  ]),
  // Output gaps: flex gap (orange), stat gap (red dash), BGP gap (dark long-dash)
  gap: mkChart('c-gap',[
    ds([],or_,[],2.0,'Gap: y − y* (NK − flex)'),
    ds([],rd,[5,3],1.6,'Gap: y − stat. trend'),
    ds([],dk,[8,4],1.3,'Gap: y − BGP'),
  ]),
  // Inflation deviations: NK (blue), zero reference (dark dash)
  pi: mkChart('c-pi',[
    ds([],bl,[],2.0,'Inflation deviation (NK)'),
    ds([],dk,[6,4],1.2,'Zero = target'),
  ]),
  // Rate deviations: nominal (blue), natural r* (orange dash)
  r: mkChart('c-r',[
    ds([],bl,[],2.0,'Nominal rate deviation'),
    ds([],or_,[6,4],1.6,'Natural rate r* deviation'),
  ]),
}};

// ── Shock UI ──────────────────────────────────────────────────────────────────
const shockState={{}};
D.shocks.forEach(sh=>{{ shockState[sh.key]={{active:false,size:1.0,rho:0}}; }});

function buildGrid() {{
  const grid=document.getElementById('shock-grid');
  D.shocks.forEach(sh=>{{
    const b=document.createElement('div');
    b.className='shock-block';
    b.innerHTML=`
      <div class="shock-head">
        <label class="tog">
          <input type="checkbox" id="chk-${{sh.key}}" onchange="toggleShock('${{sh.key}}')">
          <div class="tog-track"></div><div class="tog-thumb"></div>
        </label>
        <span class="shock-name">${{sh.label}}</span>
        <span class="shock-val" id="sv-${{sh.key}}">+1.0σ</span>
      </div>
      <div class="sl-row">
        <span>Size (σ) — negative = adverse direction</span>
        <span id="sv2-${{sh.key}}">+1.0</span>
      </div>
      <input type="range" min="-3" max="3" step="0.1" value="1"
             oninput="setSz('${{sh.key}}',this.value)">
      <div class="sl-row">
        <span>Re-application ρ (shock repeats each period, decaying)</span>
        <span id="rv-${{sh.key}}">0.00</span>
      </div>
      <input type="range" min="0" max="0.98" step="0.01" value="0"
             oninput="setRho('${{sh.key}}',this.value)">
    `;
    grid.appendChild(b);
  }});
}}

function toggleShock(k){{
  shockState[k].active=document.getElementById('chk-'+k).checked;
  update();
}}
function setSz(k,v){{
  shockState[k].size=+v;
  const s=(+v>=0?'+':'')+parseFloat(v).toFixed(1);
  document.getElementById('sv-'+k).textContent=s+'σ';
  document.getElementById('sv2-'+k).textContent=s;
  update();
}}
function setRho(k,v){{
  shockState[k].rho=+v;
  document.getElementById('rv-'+k).textContent=parseFloat(v).toFixed(2);
  update();
}}
function setFilter(f){{
  activeFilter=f;
  document.querySelectorAll('.ftab').forEach(b=>b.classList.remove('active'));
  document.getElementById('ftab-'+f).classList.add('active');
  update();
}}

// ── Main update ───────────────────────────────────────────────────────────────
function update() {{
  const T   = +document.getElementById('sl-T').value;
  document.getElementById('v-T').textContent=T;
  const Q     = Array.from({{length:T}},(_,i)=>i);
  const zeros = new Array(T).fill(0);
  const trend = Q.map(t=>TREND_Q*t);   // deterministic BGP level (log %)

  const active = D.shocks
    .filter(sh=>shockState[sh.key].active)
    .map(sh=>({{shockIdx:sh.idx, size:shockState[sh.key].size, rho:shockState[sh.key].rho}}));

  if(!active.length) {{
    // No shock selected: all lines sit on the BGP / zero-deviation baseline
    const stat0 = computeStatTrend(zeros, T);
    setChart(charts.y,   Q, [trend, trend, trend, stat0]);
    setChart(charts.gap, Q, [zeros, zeros, zeros]);
    setChart(charts.pi,  Q, [zeros, zeros]);
    setChart(charts.r,   Q, [zeros, zeros]);
    return;
  }}

  const path    = computeIRF(active, T);
  const get     = vi => path.map(s=>s[vi]);

  const y_dev   = get(VI.y);
  const yf_dev  = get(VI.yf);
  const pi_dev  = get(VI.pinf);
  const r_dev   = get(VI.r);
  const rrf_dev = get(VI.rrf);

  // Output in log-level terms (deviation from t=0 + BGP trend)
  const y_lev   = y_dev.map((v,t)=>v+trend[t]);
  const yf_lev  = yf_dev.map((v,t)=>v+trend[t]);
  const stat_tr = computeStatTrend(y_dev, T);

  // Output gaps (log %, all comparable)
  const gap_flex = y_dev.map((v,i)=>v-yf_dev[i]);   // NK minus flex (structural gap)
  const gap_stat = y_lev.map((v,i)=>v-stat_tr[i]);   // NK minus statistical trend
  const gap_bgp  = y_dev;                             // NK minus BGP (deviation from SS)

  // Inflation: deviation from target (annualised pp)
  const pi_dev_ann = pi_dev.map(v=>v*4);

  // Interest rates: deviation from SS (annualised pp)
  const r_dev_ann   = r_dev.map(v=>v*4);
  const rrf_dev_ann = rrf_dev.map(v=>v*4);

  setChart(charts.y,   Q, [trend, y_lev, yf_lev, stat_tr]);
  setChart(charts.gap, Q, [gap_flex, gap_stat, gap_bgp]);
  setChart(charts.pi,  Q, [pi_dev_ann, zeros]);
  setChart(charts.r,   Q, [r_dev_ann, rrf_dev_ann]);
}}

function setChart(ch, labels, series) {{
  ch.data.labels=labels;
  series.forEach((s,i)=>{{ if(i<ch.data.datasets.length) ch.data.datasets[i].data=s; }});
  ch.update('none');
}}

// Initialise
buildGrid();
setFilter(D.default_filter);  // activates tab + calls update()
</script>
</body>
</html>"""

with open("visual.html", "w", encoding="utf-8") as f:
    f.write(html)
print("visual.html written")


visual.html written
